# Colab Web UI Benchmark Wrapper

Use this notebook from the Colab web UI with `Runtime -> Change runtime type -> GPU`.

This notebook now lives at `cs336_systems/wrappers/colab_benchmarking_wrapper.ipynb` in the repo. It clones the repo into `/content/assignment2-systems`, installs the package in the current Colab runtime, loads the repo benchmark script, and runs a notebook-friendly wrapper around it.

In [22]:
from __future__ import annotations

import os
import subprocess
from pathlib import Path

REPO_URL = os.environ.get("ASSIGNMENT2_REPO_URL", "https://github.com/Eden-kk/assignment2-systems.git")
REPO_ROOT = Path("/content/assignment2-systems")

if REPO_ROOT.exists():
    print(f"Refreshing existing repo at {REPO_ROOT}")
    subprocess.run(["git", "fetch", "origin"], cwd=REPO_ROOT, check=True, timeout=300)
    subprocess.run(["git", "checkout", "main"], cwd=REPO_ROOT, check=True, timeout=300)
    subprocess.run(["git", "pull", "--ff-only", "origin", "main"], cwd=REPO_ROOT, check=True, timeout=300)
else:
    print(f"Cloning {REPO_URL} into {REPO_ROOT} ...")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--progress", REPO_URL, str(REPO_ROOT)],
        check=True,
        timeout=300,
    )

os.chdir(REPO_ROOT)
head = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip()
print(f"Repo root: {REPO_ROOT}")
print(f"Checked out commit: {head}")
benchmark_candidates = [
    REPO_ROOT / 'cs336_systems' / 'implementations' / 'benchmarking.py',
    REPO_ROOT / 'cs336_systems' / 'benchmarking.py',
    REPO_ROOT / 'cs336-basics' / 'scripts' / 'benchmarking.py',
]
benchmark_script = next((path for path in benchmark_candidates if path.exists()), benchmark_candidates[0])
print(f"Benchmark script: {benchmark_script}")

Refreshing existing repo at /content/assignment2-systems
Repo root: /content/assignment2-systems
Checked out commit: 2be7db5
Benchmark script: /content/assignment2-systems/cs336_systems/implementations/benchmarking.py


In [23]:
%cd /content/assignment2-systems
%pip uninstall -y -q torchaudio torchvision torchtext fastai timm
%pip install -q -e ./cs336-basics -e .

/content
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for cs336_basics (pyproject.toml) ... done
  Building editable for cs336-systems (pyproject.toml) ... done


In [24]:
import os
import shutil

import psutil
import torch


def cuda_runtime_status() -> tuple[bool, str]:
    if not torch.cuda.is_available():
        return False, "CUDA is not available in this runtime."
    try:
        _ = torch.tensor([1], device="cuda") + 1
        torch.cuda.synchronize()
        name = torch.cuda.get_device_name(0)
        capability = torch.cuda.get_device_capability(0)
        return True, f"CUDA is usable on {name} with capability {capability}."
    except Exception as exc:
        return False, f"CUDA is present but unusable with this Torch build: {exc}"


cuda_ok, cuda_message = cuda_runtime_status()
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA usable:", cuda_ok)
print("CUDA status:", cuda_message)
print("MPS available:", torch.backends.mps.is_available())
print("COLAB_RELEASE_TAG:", os.environ.get("COLAB_RELEASE_TAG"))
print("COLAB_GPU:", os.environ.get("COLAB_GPU"))

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
    print("GPU capability:", torch.cuda.get_device_capability(0))
if not cuda_ok:
    print("If Colab assigned an older GPU, reconnect until you get a newer GPU such as T4, L4, A100, or fall back to CPU.")

vm = psutil.virtual_memory()
disk = shutil.disk_usage("/")
print("RAM total (GB):", round(vm.total / 1e9, 2))
print("Disk free (GB):", round(disk.free / 1e9, 2))

Torch version: 2.6.0+cu124
CUDA available: True
CUDA usable: True
CUDA status: CUDA is usable on NVIDIA H100 80GB HBM3 with capability (9, 0).
MPS available: False
COLAB_RELEASE_TAG: release-colab-external-images_20260324-060036_RC00
COLAB_GPU: 1
GPU: NVIDIA H100 80GB HBM3
GPU count: 1
GPU capability: (9, 0)
RAM total (GB): 247.01
Disk free (GB): 197.61


In [25]:
import argparse
import importlib
import importlib.util
import sys
from pathlib import Path

import torch

repo_root = Path("/content/assignment2-systems")
basics_root = repo_root / "cs336-basics"
systems_root = repo_root / "cs336_systems"
implementations_root = systems_root / "implementations"

for path in (repo_root, basics_root, systems_root, implementations_root):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

importlib.invalidate_caches()

benchmark_candidates = [
    implementations_root / "benchmarking.py",
    systems_root / "benchmarking.py",
    basics_root / "scripts" / "benchmarking.py",
]
benchmark_path = next((path for path in benchmark_candidates if path.exists()), None)
if benchmark_path is None:
    raise FileNotFoundError(f"Could not find benchmarking.py in any expected location: {benchmark_candidates}")
spec = importlib.util.spec_from_file_location("assignment2_benchmarking", benchmark_path)
benchmarking = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = benchmarking
assert spec.loader is not None
spec.loader.exec_module(benchmarking)


def cuda_runtime_usable() -> tuple[bool, str]:
    if not torch.cuda.is_available():
        return False, "CUDA is not available in this runtime."
    try:
        _ = torch.tensor([1], device="cuda") + 1
        torch.cuda.synchronize()
        name = torch.cuda.get_device_name(0)
        capability = torch.cuda.get_device_capability(0)
        return True, f"Using CUDA on {name} with capability {capability}."
    except Exception as exc:
        return False, f"Falling back from CUDA because this runtime cannot execute kernels with the installed Torch build: {exc}"


def detect_device(preferred: str = "auto") -> str:
    if preferred != "auto":
        return preferred
    cuda_ok, _cuda_message = cuda_runtime_usable()
    if cuda_ok:
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def make_config(
    *,
    model_size: str,
    context_length: int,
    batch_size: int,
    vocab_size: int,
    rope_theta: float,
    warmup_steps: int,
    measurement_steps: int,
    mode: str,
    preferred_device: str,
    dtype: str,
    precision_autocast: str,
    seed: int,
    optimizer_lr: float = 1e-3,
    memory_snapshot_path: str | None = None,
    report_peak_memory: bool = False,
):
    args = argparse.Namespace(
        model_size=model_size,
        context_length=context_length,
        batch_size=batch_size,
        vocab_size=vocab_size,
        rope_theta=rope_theta,
        warmup_steps=warmup_steps,
        measurement_steps=measurement_steps,
        mode=mode,
        device=detect_device(preferred_device),
        dtype=dtype,
        precision_autocast=precision_autocast,
        optimizer_lr=optimizer_lr,
        memory_snapshot_path=memory_snapshot_path,
        report_peak_memory=report_peak_memory,
        seed=seed,
    )
    return benchmarking.resolve_config(args)


def benchmark_model(config):
    torch.manual_seed(config.seed)
    model = benchmarking.build_model(config)
    batch = benchmarking.make_batch(config)
    step_times = benchmarking.benchmark_steps(model, batch, config)
    summary = benchmarking.summarize_timings(step_times)
    return step_times, summary


print("Loaded benchmark module from:", benchmark_path)

Loaded benchmark module from: /content/assignment2-systems/cs336_systems/implementations/benchmarking.py


In [26]:
PREFERRED_DEVICE = "auto"  # "auto", "cuda", "mps", or "cpu"
MODEL_SIZE = "small"
CONTEXT_LENGTH = 128
BATCH_SIZE = 4
VOCAB_SIZE = 10_000
ROPE_THETA = 10_000.0
WARMUP_STEPS = 5
MEASUREMENT_STEPS = 10
MODE = "forward"  # "forward" or "forward-backward"
DTYPE = "float32"  # parameter dtype: "float16", "float32", or "bfloat16"
AUTOCAST_CONFIGS = ["none", "float16", "bfloat16"]
SEED = 0

selected_device = detect_device(PREFERRED_DEVICE)
print("Selected device:", selected_device)
print("Autocast sweep:", AUTOCAST_CONFIGS)

Selected device: cuda
Autocast sweep: ['none', 'float16', 'bfloat16']


In [13]:
MODEL_SIZE_SWEEP = ["small", "medium", "large", "xl"]
AUTOCAST_CONFIGS = ["none", "float16", "bfloat16"]

sweep_rows = []

for model_size in MODEL_SIZE_SWEEP:
    for autocast_name in AUTOCAST_CONFIGS:
        try:
            config = make_config(
                model_size=model_size,
                context_length=CONTEXT_LENGTH,
                batch_size=BATCH_SIZE,
                vocab_size=VOCAB_SIZE,
                rope_theta=ROPE_THETA,
                warmup_steps=WARMUP_STEPS,
                measurement_steps=MEASUREMENT_STEPS,
                mode=MODE,
                preferred_device=PREFERRED_DEVICE,
                dtype=DTYPE,
                precision_autocast=autocast_name,
                seed=SEED,
            )
            _step_times, summary = benchmark_model(config)
            sweep_rows.append(
                {
                    "model_size": model_size,
                    "autocast": autocast_name,
                    "device": config.device,
                    "dtype": str(config.dtype),
                    "precision_autocast": str(config.precision_autocast),
                    "mean_ms": summary["mean"] * 1000,
                    "stdev_ms": summary["stdev"] * 1000,
                    "min_ms": summary["min"] * 1000,
                    "max_ms": summary["max"] * 1000,
                    "status": "ok",
                }
            )
        except Exception as exc:
            sweep_rows.append(
                {
                    "model_size": model_size,
                    "autocast": autocast_name,
                    "device": selected_device,
                    "dtype": DTYPE,
                    "precision_autocast": autocast_name,
                    "mean_ms": None,
                    "stdev_ms": None,
                    "min_ms": None,
                    "max_ms": None,
                    "status": f"failed: {exc}",
                }
            )

sweep_df = pd.DataFrame(sweep_rows)
sweep_df


,model_size,autocast,device,dtype,precision_autocast,mean_ms,stdev_ms,min_ms,max_ms,status
0,small,none,cuda,torch.float32,None,17.650455,0.418504,17.064979,18.280434,ok
1,small,float16,cuda,torch.float32,torch.float16,20.879834,0.710132,19.840776,21.939289,ok
2,small,bfloat16,cuda,torch.float32,torch.bfloat16,20.561738,0.451032,19.916666,21.278742,ok
3,medium,none,cuda,torch.float32,None,36.648975,1.175828,34.957884,38.131927,ok
4,medium,float16,cuda,torch.float32,torch.float16,37.893757,1.184740,36.228726,39.452161,ok
5,medium,bfloat16,cuda,torch.float32,torch.bfloat16,39.293089,1.444523,36.671624,41.294531,ok
6,large,none,cuda,torch.float32,None,56.220984,2.020819,53.104632,60.065268,ok
7,large,float16,cuda,torch.float32,torch.float16,61.143611,2.675043,57.243206,65.740386,ok
8,large,bfloat16,cuda,torch.float32,torch.bfloat16,57.956793,1.577391,55.541945,61.436267,ok
9,xl,none,cuda,torch.float32,None,72.149408,2.203892,68.903489,75.603007,ok


In [14]:
pivot_df = sweep_df.pivot(index="model_size", columns="autocast", values="mean_ms")
pivot_df


autocast,bfloat16,float16,none
model_size,,,
large,57.956793,61.143611,56.220984
medium,39.293089,37.893757,36.648975
small,20.561738,20.879834,17.650455
xl,77.418463,78.992241,72.149408


# Memory Profiling

These cells drive the `memory_profiling` part of the assignment using the memory hooks added to `cs336_systems/implementations/benchmarking.py`. Run them on a CUDA runtime only.


In [30]:
MEMORY_MODEL_SIZE = "2.7B"
MEMORY_CONTEXT_LENGTH = 128
MEMORY_CONTEXT_LENGTH_SWEEP = [128, 256, 512]
MEMORY_MODE = "forward"  # "forward" or "train-step"
MEMORY_DTYPE = "float32"
MEMORY_AUTOCAST = "none"  # use "bfloat16" for mixed-precision memory experiments
MEMORY_SNAPSHOT_PATH = f"/content/memory_{MEMORY_MODEL_SIZE}_{MEMORY_MODE}_ctx{MEMORY_CONTEXT_LENGTH}_{MEMORY_AUTOCAST}.pickle"
MEMORY_OPTIMIZER_LR = 1e-3

if selected_device != "cuda":
    raise RuntimeError("Memory profiling cells require a CUDA runtime.")

print("Snapshot path:", MEMORY_SNAPSHOT_PATH)
print("Context sweep:", MEMORY_CONTEXT_LENGTH_SWEEP)


Snapshot path: /content/memory_2.7B_forward_ctx128_none.pickle
Context sweep: [128, 256, 512]


In [31]:
memory_config = make_config(
    model_size=MEMORY_MODEL_SIZE,
    context_length=MEMORY_CONTEXT_LENGTH,
    batch_size=BATCH_SIZE,
    vocab_size=VOCAB_SIZE,
    rope_theta=ROPE_THETA,
    warmup_steps=WARMUP_STEPS,
    measurement_steps=MEASUREMENT_STEPS,
    mode=MEMORY_MODE,
    preferred_device="cuda",
    dtype=MEMORY_DTYPE,
    precision_autocast=MEMORY_AUTOCAST,
    seed=SEED,
    optimizer_lr=MEMORY_OPTIMIZER_LR,
    memory_snapshot_path=MEMORY_SNAPSHOT_PATH,
    report_peak_memory=True,
)

torch.manual_seed(memory_config.seed)
memory_model = benchmarking.build_model(memory_config)
memory_optimizer = benchmarking.build_optimizer(memory_model, memory_config)
memory_batch = benchmarking.make_batch(memory_config)

# Warm up before recording memory history.
for _ in range(memory_config.warmup_steps):
    benchmarking.run_step(memory_model, memory_batch, memory_config, memory_optimizer)

memory_stats = benchmarking.maybe_profile_memory(memory_model, memory_batch, memory_config, memory_optimizer)
print(memory_config)
print(memory_stats)
print({"memory_snapshot_path": MEMORY_SNAPSHOT_PATH})


BenchmarkConfig(model_size='2.7B', context_length=128, batch_size=4, vocab_size=10000, rope_theta=10000.0, warmup_steps=5, measurement_steps=10, mode='forward', device='cuda', dtype=torch.float32, precision_autocast=None, optimizer_lr=0.001, memory_snapshot_path='/content/memory_2.7B_forward_ctx128_none.pickle', report_peak_memory=True, seed=0)
{'peak_allocated_mb': 26228.4853515625, 'peak_reserved_mb': 45208.0}
{'memory_snapshot_path': '/content/memory_2.7B_forward_ctx128_none.pickle'}


In [ ]:
memory_rows = []

for context_length in MEMORY_CONTEXT_LENGTH_SWEEP:
    sweep_config = make_config(
        model_size=MEMORY_MODEL_SIZE,
        context_length=context_length,
        batch_size=BATCH_SIZE,
        vocab_size=VOCAB_SIZE,
        rope_theta=ROPE_THETA,
        warmup_steps=WARMUP_STEPS,
        measurement_steps=MEASUREMENT_STEPS,
        mode=MEMORY_MODE,
        preferred_device="cuda",
        dtype=MEMORY_DTYPE,
        precision_autocast=MEMORY_AUTOCAST,
        seed=SEED,
        optimizer_lr=MEMORY_OPTIMIZER_LR,
        memory_snapshot_path=None,
        report_peak_memory=True,
    )

    torch.manual_seed(sweep_config.seed)
    sweep_model = benchmarking.build_model(sweep_config)
    sweep_optimizer = benchmarking.build_optimizer(sweep_model, sweep_config)
    sweep_batch = benchmarking.make_batch(sweep_config)

    for _ in range(sweep_config.warmup_steps):
        benchmarking.run_step(sweep_model, sweep_batch, sweep_config, sweep_optimizer)

    peak_stats = benchmarking.maybe_profile_memory(sweep_model, sweep_batch, sweep_config, sweep_optimizer)
    memory_rows.append(
        {
            "context_length": context_length,
            "mode": MEMORY_MODE,
            "autocast": MEMORY_AUTOCAST,
            "peak_allocated_mb": peak_stats["peak_allocated_mb"],
            "peak_reserved_mb": peak_stats["peak_reserved_mb"],
        }
    )

memory_df = pd.DataFrame(memory_rows)
memory_df
